In [16]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict, List, Optional
from langchain_huggingface import HuggingFacePipeline , HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import operator

In [3]:
load_dotenv()  # take environment variables from .env.

True

In [2]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    temperature=0.7,
)

model = ChatHuggingFace(llm=llm)

d:\LangGarph-tutorials\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
class WorkflowState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [7]:
def generate_joke(state: WorkflowState) -> WorkflowState:
    prompt = f"Tell me a joke about {state['topic']}"
    response = model.invoke(prompt).content
    return {"joke": response}

In [9]:
def explain_joke(state:WorkflowState)->WorkflowState:
    prompt=f"Explain the this Joke: {state['joke']}"
    response=model.invoke(prompt).content
    return {"explanation":response}

In [18]:
stategraph=StateGraph(WorkflowState)

checkpoint_saver=InMemorySaver()
stategraph.add_node("generate_joke", generate_joke)
stategraph.add_node("explain_joke",explain_joke)

stategraph.add_edge(START, "generate_joke")
stategraph.add_edge("generate_joke", "explain_joke")
stategraph.add_edge("explain_joke", END)


workflow=stategraph.compile(checkpointer=checkpoint_saver)

In [20]:
config = {"configurable": {"thread_id": "1"}}
initial_state={"topic":"programming"}
result=workflow.invoke(initial_state ,config=config)
print("result:",result['joke'])
print("result:",result['explanation'])

result: Sure! Here's a light-hearted programming joke for you:

Why did the programmer quit his job?

Because he didn't get arrays.
result: This joke is a play on words and a common stereotype about programmers. The punchline, "Because he didn't get arrays," is a humorous reference to the fact that the programmer couldn't understand or work with arrays, which are fundamental data structures in computer programming used to store multiple values in a single variable.

In programming, an array is a collection of elements of the same type stored and accessed using a single name and an index. If a programmer "doesn't get arrays," it suggests a basic misunderstanding of a core concept in programming, which could lead to frustration and, in the context of the joke, could be a reason for quitting the job.

The joke also plays on the idea that some people might not understand or appreciate the nuances of programming concepts, using this to create a relatable but humorous scenario.


In [21]:
workflow.get_state(config=config)

StateSnapshot(values={'topic': 'programming', 'joke': "Sure! Here's a light-hearted programming joke for you:\n\nWhy did the programmer quit his job?\n\nBecause he didn't get arrays.", 'explanation': 'This joke is a play on words and a common stereotype about programmers. The punchline, "Because he didn\'t get arrays," is a humorous reference to the fact that the programmer couldn\'t understand or work with arrays, which are fundamental data structures in computer programming used to store multiple values in a single variable.\n\nIn programming, an array is a collection of elements of the same type stored and accessed using a single name and an index. If a programmer "doesn\'t get arrays," it suggests a basic misunderstanding of a core concept in programming, which could lead to frustration and, in the context of the joke, could be a reason for quitting the job.\n\nThe joke also plays on the idea that some people might not understand or appreciate the nuances of programming concepts, u

In [23]:
list(workflow.get_state_history(config=config))

[StateSnapshot(values={'topic': 'programming', 'joke': "Sure! Here's a light-hearted programming joke for you:\n\nWhy did the programmer quit his job?\n\nBecause he didn't get arrays.", 'explanation': 'This joke is a play on words and a common stereotype about programmers. The punchline, "Because he didn\'t get arrays," is a humorous reference to the fact that the programmer couldn\'t understand or work with arrays, which are fundamental data structures in computer programming used to store multiple values in a single variable.\n\nIn programming, an array is a collection of elements of the same type stored and accessed using a single name and an index. If a programmer "doesn\'t get arrays," it suggests a basic misunderstanding of a core concept in programming, which could lead to frustration and, in the context of the joke, could be a reason for quitting the job.\n\nThe joke also plays on the idea that some people might not understand or appreciate the nuances of programming concepts, 

In [25]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f197181-45b2-60db-8000-d48360b41766"}})

StateSnapshot(values={'topic': 'programming'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f197181-45b2-60db-8000-d48360b41766'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-13T13:08:30.172162+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f197181-45a9-66d8-bfff-f56ca85acce1'}}, tasks=(PregelTask(id='e97fd714-ae35-924d-be4b-53282b2cb175', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Sure! Here's a light-hearted programming joke for you:\n\nWhy did the programmer quit his job?\n\nBecause he didn't get arrays."}),), interrupts=())

In [28]:
list(workflow.get_state_history({"configurable": {"thread_id": "1", "checkpoint_id": "1f197181-4d09-65e7-8001-5b4362c1eafb"}}))

[StateSnapshot(values={'topic': 'programming', 'joke': "Sure! Here's a light-hearted programming joke for you:\n\nWhy did the programmer quit his job?\n\nBecause he didn't get arrays."}, next=('explain_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f197181-4d09-65e7-8001-5b4362c1eafb'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-13T13:08:30.941919+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f197181-45b2-60db-8000-d48360b41766'}}, tasks=(PregelTask(id='ab724cf3-6ec5-78ea-298e-4e9d00a2959e', name='explain_joke', path=('__pregel_pull', 'explain_joke'), error=None, interrupts=(), state=None, result={'explanation': 'This joke is a play on words and a common stereotype about programmers. The punchline, "Because he didn\'t get arrays," is a humorous reference to the fact that the programmer couldn\'t understand or work with arrays, which are fundamental data stru